# Instrument profiles

Nominal Gaussian resolution matching with offline NIRSpec calibration data.
This synthetic example models neither throughput nor detector noise. The model
has real source coverage outside the final detector limits for convolution.


In [ ]:
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from speclib import Spectrum, get_profile, list_instrument_profiles

wave = np.geomspace(0.5, 6.0, 30000) * u.um
x = wave.to_value(u.um)
flux = np.ones(wave.size)
for center in (1.2, 3.5, 4.5):
    flux += 4 * np.exp(-0.5 * ((x - center) / 0.0004)**2)
unit = u.erg / (u.s * u.cm**2 * u.AA)
source = Spectrum(spectral_axis=wave, flux=flux * unit)
list_instrument_profiles()


## Coverage, resolving power and sampling

Every result is an ordinary `Spectrum`; the default preserves valid source samples.
`valid_intervals` declares missing coverage explicitly, without artificial flux
samples or treating detector regions as independent spectral orders.
Automatic centers are not detector pixels. Resampling integrates implicit bins;
coarse bin integration and sparse sampling can change apparent line widths.
Preserve fine native sampling or request more centers for width measurements.
Explicit centers and all their bin edges must stay within valid coverage.


In [ ]:
profile = get_profile("JWST/NIRSpec/PRISM/CLEAR")
print(profile.configuration)
print(profile.valid_intervals)
print(profile.calibration_version)
print(profile.resolution_definition)
print(profile.assumptions)
print(profile.provenance["url"])  # Original FITS, independent of coverage source.
print(profile.provenance["sha256"])
print(profile.coverage_source_url)
assert profile.provenance["filename"] == "jwst_nirspec_prism_disp.fits"

prism = source.observed_by(profile)
sampled = source.observed_by(profile, samples_per_resolution_element=3)
explicit_centers = np.linspace(0.65, 5.2, 300) * u.um
explicit = source.observed_by(profile, wavelength=explicit_centers)
assert len(prism.flux) > len(sampled.flux)
assert np.array_equal(explicit.wavelength, explicit_centers)
print(prism.valid_intervals)
continuous = prism


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(profile.wavelength, profile.resolving_power)
axes[0].set(xlabel="Wavelength (micrometers)", ylabel="Nominal resolving power", title="NIRSpec PRISM/CLEAR")
axes[1].plot(source.wavelength.to_value(u.um), source.flux.value, label="Source")
axes[1].plot(continuous.wavelength.to_value(u.um), continuous.flux.value, label="Broadened")
axes[1].plot(sampled.wavelength.to_value(u.um), sampled.flux.value, ".", label="3 samples/FWHM")
axes[1].set(xlim=(1.15, 1.25), xlabel="Wavelength (micrometers)", ylabel="Flux density (arbitrary units)")
axes[1].legend()
fig.tight_layout()
plt.show()


## Detector gaps

G395H/F290LP returns one ordinary spectrum with explicit `valid_intervals`.
Its arrays contain no samples in the 3.72–3.82 micrometer gap. Coverage is
approximate BOTS/S1600A1/SUB2048 geometry, not a visit-specific wavelength solution.
Source flux in the output gap still contributes to convolution. Plot each valid
interval separately; resampling and binning reject requests crossing the gap.

For a preloaded high-resolution `SpectralGrid` with sufficient source coverage,
`observed_grid = grid.observed_by("jwst/nirspec/g395h/f290lp")` matches every model.
Then `observed_grid.get_spectrum(teff, logg, metallicity)` returns an independent
`Spectrum` retaining the same detector coverage and gap-aware operations.
`get_flux` instead returns only the interpolated flux Quantity. See the
spectral-grids notebook for executable examples of both retrieval methods.


In [ ]:
gapped = source.observed_by("jwst/nirspec/g395h/f290lp")
print(gapped.valid_intervals)
fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
for lower, upper in gapped.valid_intervals:
    selected = (gapped.spectral_axis >= lower) & (gapped.spectral_axis <= upper)
    ax.plot(gapped.spectral_axis[selected].to_value(u.um), gapped.flux[selected].value)
ax.axvspan(3.72, 3.82, color="0.9", label="Detector gap")
ax.set(xlabel="Wavelength (micrometers)", ylabel="Flux density (arbitrary units)")
ax.legend()
plt.show()
assert not np.any((gapped.wavelength > 3.72*u.um) & (gapped.wavelength < 3.82*u.um))
centers = np.r_[np.linspace(3.0, 3.6, 20), np.linspace(4., 5., 20)]*u.um
resampled = gapped.resample(centers)
assert len(resampled.valid_intervals) == 2
selected = gapped.select_wavelength(4.*u.um, 5.*u.um)
assert len(selected.valid_intervals) == 1
try:
    gapped.resample([3.76, 3.77, 3.78]*u.um)
except ValueError as error:
    print(error)
else:
    raise AssertionError("Gap request was not rejected")
assert source.wavelength.min() == 0.5*u.um
assert "instrument_profile" not in source.meta


## Calibration and limitations

The [NASA/STScI resolving-power tables](https://jwst-docs.stsci.edu/jwst-near-infrared-spectrograph/nirspec-instrumentation/nirspec-dispersers-and-filters)
use a 2.2-pixel resolution element, here interpreted as a Gaussian FWHM. The
[BOTS coverage table](https://jwst-docs.stsci.edu/jwst-near-infrared-spectrograph/nirspec-operations/nirspec-bots-operations/nirspec-bots-wavelength-ranges-and-gaps)
sets the output intervals. These nominal disperser curves do not reproduce
measured wide-slit point-source LSFs or non-Gaussian wings. Intrinsic source
widths are assumed negligible; sampling checks cannot establish physical model
resolution. See the instrument-profiles user guide for padding, resampling,
provenance and source-resolution requirements.

The [scientific provenance reference](../user_guide/instrument_provenance.rst)
links each original file and records its creation/history, conversion and source
limitations. `profile.provenance` exposes that manifest record offline. The
nominal tables are dispersion-derived ground/modeling prescriptions; adopting
their 2.2-pixel element as a Gaussian FWHM is Speclib's approximation, not an
empirical in-flight LSF measurement.
